## Archived optional extension — outside final project scope (12 November 2026)

**Do not run this notebook as part of the completed submission.** The adopted study ended after eight verified full-epoch experiments. The additional 16 training combinations were proposed for future robustness testing but were never carried out or required for the frozen project. The completed results and submission files are indexed in [`README.md`](README.md).

---

# Complete the remaining 16 full-epoch experiments — seeds 20261029 and 20261030

The first eight 30-true-epoch runs are already completed: sizes 70, 140, 280, 420 × losses Cross-Entropy and Poly-1, seed 20261028. This notebook runs the **remaining 16** combinations for the other two seeds, keeping exactly the same frozen 180-image *train-only* validation holdout.

**GPU quota protection:** By default it finishes **at most two NEW runs per Colab session** (`MAX_NEW_RUNS_PER_SESSION = 2`). After each pair, you can disconnect and later open this notebook and **Runtime → Run all** again: finished runs are checked and skipped, partially completed ones resume from the last saved full epoch. You can change the limit if you have extra GPU access.

Every run has exactly **30 complete passes over its 70/140/280/420 image training set**, with steps 540/1050/2100/3150 per run. The earlier fixed-216-step study remains untouched. No old training, validation, original report, or official test outputs are overwritten.

**To run:** Choose **Runtime → Change runtime type → T4 GPU**. Then **Runtime → Run all**, authorize the existing Google Drive workspace, and let the notebook finish its current pair of runs. All checkpoints and logs persist in `MyDrive/Nazih_Coursework/06_Full_Epoch_Pilot/`. Never run this notebook in two concurrent sessions, as both might attempt to write the same checkpoints. The previously inspected official TEST set is not evaluated.


In [ ]:
import torch
import torchvision
assert torch.cuda.is_available(), "GPU unavailable: Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__, "| torchvision:", torchvision.__version__)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import subprocess, sys, os, shutil, hashlib, json
REPOSITORY = "https://github.com/ID9999999999999/DL-Computer-vision.git"
BRANCH = "nazih-training-set-size-20261109"
REPO = Path("/content/nazih_full_epoch_project")
PROJECT = REPO / "COURSEWORK_NAZIH"
DRIVE_BASE = Path("/content/drive/MyDrive/Nazih_Coursework")
DRIVE_OUT = DRIVE_BASE / "06_Full_Epoch_Pilot"
DRIVE_DATASET = DRIVE_BASE / "05_Official_Dataset" / "KolektorSDD2.zip"

assert DRIVE_BASE.is_dir(), f"Existing project Drive folder not visible: {DRIVE_BASE}"
assert DRIVE_OUT.is_dir(), f"Expected backup folder missing (created earlier): {DRIVE_OUT}"
assert DRIVE_DATASET.is_file(), f"Official ZIP missing on Drive: {DRIVE_DATASET}"

if not (REPO / ".git").is_dir():
    assert not REPO.exists(), "Refusing to overwrite unrelated /content project"
    subprocess.run(["git", "clone", "--single-branch", "--branch", BRANCH,
                    "--depth", "1", REPOSITORY, str(REPO)], check=True)
else:
    status = subprocess.check_output(
        ["git", "-C", str(REPO), "status", "--porcelain"], text=True
    ).strip()
    assert not status, "Unexpected local changes in Colab clone"
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH], check=True)

assert (PROJECT / "scripts" / "full_epoch_pilot.py").is_file()
assert (PROJECT / "protocol.json").is_file()
output_link = PROJECT / "data" / "full_epoch_pilot"
output_link.parent.mkdir(parents=True, exist_ok=True)
if output_link.is_symlink():
    assert output_link.resolve() == DRIVE_OUT.resolve(), "Output points to the wrong Drive folder!"
elif output_link.exists():
    raise RuntimeError("Refusing to overwrite existing non-symlink output directory")
else:
    output_link.symlink_to(DRIVE_OUT, target_is_directory=True)

print("✅ Code cloned and persistent Drive output linked")
print("✅ Existing historical results remain untouched")
print("Output:", output_link.resolve())

In [ ]:
EXPECTED_SHA256 = "edcdb486809b24f1d17b785e30c52fafc5999554dd5fe18ddf77b61ceb6f36a8"
LOCAL_DATASET = Path("/content/KolektorSDD2.zip")
def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for piece in iter(lambda: f.read(4 * 1024 * 1024), b""):
            h.update(piece)
    return h.hexdigest()

if not LOCAL_DATASET.exists():
    free = shutil.disk_usage("/content").free
    assert free > DRIVE_DATASET.stat().st_size + 1024**3, "Not enough Colab disk space"
    shutil.copy2(DRIVE_DATASET, LOCAL_DATASET)

assert sha256(LOCAL_DATASET) == EXPECTED_SHA256, "Official dataset SHA256 differs. Stop."
print("✅ Dataset copied from Google Drive and SHA256 verified")
print("ZIP bytes:", LOCAL_DATASET.stat().st_size)

In [ ]:
SCRIPT = PROJECT / "scripts" / "full_epoch_pilot.py"
SEEDS = (20261029, 20261030)
SIZES = (70, 140, 280, 420)
LOSSES = ("ce", "poly1")
TOTAL_EPOCHS = 30
RUNS = [(seed, size, loss) for seed in SEEDS for size in SIZES for loss in LOSSES]
MAX_NEW_RUNS_PER_SESSION = 2  # Lower GPU quota use; increase only if available

assert len(RUNS) == 16

# Verify the 8 original full-epoch runs were retained.
for size in SIZES:
    for loss in LOSSES:
        previous = DRIVE_OUT / f"n{size}_seed20261028_{loss}" / "COMPLETE.json"
        assert previous.is_file(), f"An earlier verified run is missing: {previous}"
print("✅ All eight original full-epoch runs retained")

REFERENCE_SPLIT = DRIVE_OUT / "n420_seed20261028_ce" / "split.json"
assert REFERENCE_SPLIT.is_file()
assert sha256(REFERENCE_SPLIT) == "1be0f86837238f81495121ae485b31d65ddf2dce512f20c540734057e516bcb0", (
    "Previously frozen reference split changed!"
)
reference = json.loads(REFERENCE_SPLIT.read_text(encoding="utf-8"))
reference_validation = reference["validation_rows"]
assert len(reference_validation) == 180
assert sum(row["label"] == 0 for row in reference_validation) == 150
assert sum(row["label"] == 1 for row in reference_validation) == 30

import sys as _sys
_sys.path.insert(0, str(SCRIPT.parent))
from full_epoch_pilot import fixed_splits
for n in SIZES:
    training, validation, frozen_sha = fixed_splits(LOCAL_DATASET, n)
    assert validation == reference_validation, f"Validation changed at size {n}"
    assert frozen_sha == reference["manifest_sha256"]
    assert len(training) == n
    assert not {row["sha256"] for row in training} & {
        row["sha256"] for row in validation
    }
    print(f"✅ n={n}: training {len(training)}, validation 180, no overlap")

print("✅ DATA PREFLIGHT PASSED — no official test images opened")
print("Pending design: 16 runs, two full seeds, 30 true epochs each")

## Execute only a small resumable GPU batch

A run is considered complete only when its `COMPLETE.json` exists, has exactly 30 recorded full epochs, and its best checkpoint matches its recorded SHA256. Interrupted runs resume from the previous saved epoch. The notebook will stop after `MAX_NEW_RUNS_PER_SESSION` completed new runs; rerun all cells in a later Colab T4 session to continue. If Colab interrupts in the middle of a run, just rerun this notebook; do not delete its progress files.


In [ ]:
done_runs = []
pending_runs = []
new_runs_completed = 0

for seed, size, loss in RUNS:
    name = f"n{size}_seed{seed}_{loss}"
    run_dir = DRIVE_OUT / name
    complete_path = run_dir / "COMPLETE.json"
    args = [
        "--archive", str(LOCAL_DATASET),
        "--size", str(size),
        "--seed", str(seed),
        "--loss", loss,
        "--epochs", str(TOTAL_EPOCHS),
        "--batch-size", "4",
        "--device", "cuda",
    ]

    if not complete_path.exists():
        if new_runs_completed >= MAX_NEW_RUNS_PER_SESSION:
            pending_runs.append(name)
            continue
        print("\nSTART OR RESUME:", name, flush=True)
        subprocess.run([_sys.executable, str(SCRIPT), *args], cwd=PROJECT, check=True)
        assert complete_path.is_file(), f"Training did not complete: {name}"
        new_runs_completed += 1
    else:
        print("PRESERVED:", name)

    record = json.loads(complete_path.read_text(encoding="utf-8"))
    history = record["history"]
    expected_steps = (size + 3) // 4

    assert record["identity"]["size"] == size
    assert record["identity"]["seed"] == seed
    assert record["identity"]["loss"] == loss
    assert record["identity"]["epochs"] == TOTAL_EPOCHS
    assert len(history) == TOTAL_EPOCHS
    assert [x["full_epoch"] for x in history] == list(range(1, TOTAL_EPOCHS + 1))
    assert all(x["training_samples_seen"] == size for x in history)
    assert all(x["training_optimizer_steps"] == expected_steps for x in history)
    assert record["identity"]["total_planned_optimizer_steps"] == expected_steps * TOTAL_EPOCHS
    assert record["official_test_evaluated"] is False
    assert sha256(run_dir / "best_model.pt") == record["best_model_sha256"]
    assert sha256(run_dir / "split.json") == record["split_sha256"]

    split = json.loads((run_dir / "split.json").read_text(encoding="utf-8"))
    assert split["validation_rows"] == reference_validation

    done_runs.append({
        "seed": seed, "size": size, "loss": loss,
        "best_epoch": record["best_epoch"],
        "macro_f1": record["best_validation_macro_f1"],
        "total_steps": expected_steps * TOTAL_EPOCHS,
    })
    print("✅ VERIFIED:", name, "best epoch:", record["best_epoch"],
          "Val Macro-F1:", round(record["best_validation_macro_f1"], 5))

print("\n===== FULL-EPOCH STUDY STATUS =====")
print("New complete this session:", new_runs_completed)
print("Remaining-seed runs complete:", len(done_runs), "/ 16")
print("Entire 30-epoch study:", 8 + len(done_runs), "/ 24")
print("Still pending:", len(pending_runs))
if pending_runs:
    print("Next:", pending_runs[0])
print("✅ All already completed runs preserved on Google Drive")

In [ ]:
from pprint import pprint
pprint(done_runs)

# Read-only tally of the full 24-run design; don't edit past results.
all_runs = []
for seed in (20261028, 20261029, 20261030):
    for size in SIZES:
        for loss in LOSSES:
            name = f"n{size}_seed{seed}_{loss}"
            path = DRIVE_OUT / name / "COMPLETE.json"
            if not path.exists():
                continue
            item = json.loads(path.read_text(encoding="utf-8"))
            assert len(item["history"]) == 30
            assert item["official_test_evaluated"] is False
            all_runs.append({
                "seed": seed, "size": size, "loss": loss,
                "best_epoch": item["best_epoch"],
                "macro_f1": item["best_validation_macro_f1"],
            })
print(f"✅ Confirmed on Drive: {len(all_runs)}/24 real 30-full-epoch runs")
if len(all_runs) == 24:
    print("✅ ALL 24 FULL-EPOCH EXPERIMENTS COMPLETE")
else:
    print(f"Further 30-epoch trainings needed: {24 - len(all_runs)}")
    print("To continue, reopen this notebook on Colab T4 and Run all.")

In [ ]:
import matplotlib.pyplot as plt
if all_runs:
    plt.figure(figsize=(8, 4))
    for seed in (20261028, 20261029, 20261030):
        for loss in LOSSES:
            subset = sorted(
                (r for r in all_runs if r["seed"] == seed and r["loss"] == loss),
                key=lambda r: r["size"]
            )
            if subset:
                plt.plot(
                    [r["size"] for r in subset],
                    [100 * r["macro_f1"] for r in subset],
                    marker="o",
                    label=f"seed={seed} {loss.upper()}"
                )
    plt.title("Exploratory internal validation after 30 full epochs")
    plt.xlabel("Training images")
    plt.ylabel("Best validation Macro-F1 (%)")
    plt.grid(alpha=0.25)
    plt.legend(fontsize=8, ncol=2)
    plt.show()

print("Original 216-step experiment untouched; official TEST not used in new selection.")